In [2]:
from dotenv import load_dotenv
load_dotenv()

import uuid
from typing import List
from pydantic import BaseModel, Field

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage
from langchain_core.runnables import RunnableConfig

from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.store.postgres import PostgresStore
from langgraph.store.base import BaseStore

In [4]:
SYSTEM_PROMPT_TEMPLATE = """You are a helpful assistant with memory capabilities.
If user-specific memory is available, use it to personalize 
your responses based on what you know about the user.

Your goal is to provide relevant, friendly, and tailored 
assistance that reflects the user’s preferences, context, and past interactions.

If the user’s name or relevant personal context is available, always personalize your responses by:
    – Always Address the user by name (e.g., "Sure, amir...") when appropriate
    – Referencing known projects, tools, or preferences (e.g., "your MCP server python based project")
    – Adjusting the tone to feel friendly, natural, and directly aimed at the user

Avoid generic phrasing when personalization is possible.

Use personalization especially in:
    – Greetings and transitions
    – Help or guidance tailored to tools and frameworks the user uses
    – Follow-up messages that continue from past context

Always ensure that personalization is based only on known user details and not assumed.

In the end suggest 3 relevant further questions based on the current response and user profile

The user’s memory (which may be empty) is provided as: {user_details_content}
"""

In [5]:
llm = ChatGroq(model_name="llama-3.3-70b-versatile")

In [6]:
class MemoryItem(BaseModel):
    text: str = Field(description="Atomic user memory")
    is_new: bool = Field(description="True if new, false if duplicate")

In [7]:
class MemoryDecision(BaseModel):
    should_write: bool
    memories: List[MemoryItem] = Field(default_factory=list)

In [8]:
memory_extractor = llm.with_structured_output(MemoryDecision)

In [9]:
MEMORY_PROMPT = """You are responsible for updating and maintaining accurate user memory.

CURRENT USER DETAILS (existing memories):
{user_details_content}

TASK:
- Review the user's latest message.
- Extract user-specific info worth storing long-term (identity, stable preferences, ongoing projects/goals).
- For each extracted item, set is_new=true ONLY if it adds NEW information compared to CURRENT USER DETAILS.
- If it is basically the same meaning as something already present, set is_new=false.
- Keep each memory as a short atomic sentence.
- No speculation; only facts stated by the user.
- If there is nothing memory-worthy, return should_write=false and an empty list.
"""

In [10]:

def remember_node(state: MessagesState, config: RunnableConfig, *, store: BaseStore):
    user_id = config["configurable"]["user_id"]
    ns = ("user", user_id, "details")
    items = store.search(ns)
    existing = "\n".join(it.value.get("data", "") for it in items) if items else "(empty)"

    last_text = state["messages"][-1].content

    decision: MemoryDecision = memory_extractor.invoke(
        [
            SystemMessage(content=MEMORY_PROMPT.format(user_details_content=existing)),
            {"role": "user", "content": last_text},
        ]
    )

    if decision.should_write:
        for mem in decision.memories:
            if mem.is_new and mem.text.strip():
                store.put(ns, str(uuid.uuid4()), {"data": mem.text.strip()})

    return {}

In [11]:
def chat_node(state: MessagesState, config: RunnableConfig, *, store: BaseStore):
    user_id = config["configurable"]["user_id"]
    ns = ("user", user_id, "details")

    items = store.search(ns)
    user_details = "\n".join(it.value.get("data", "") for it in items) if items else ""

    system_msg = SystemMessage(
        content=SYSTEM_PROMPT_TEMPLATE.format(user_details_content=user_details or "(empty)")
    )

    response = llm.invoke([system_msg] + state["messages"])
    return {"messages": [response]}

In [12]:

builder = StateGraph(MessagesState)
builder.add_node("remember", remember_node)
builder.add_node("chat", chat_node)
builder.add_edge(START, "remember")
builder.add_edge("remember", "chat")
builder.add_edge("chat", END)

In [20]:

import os

DB_URI = os.getenv("DATABASE_URL")

with PostgresStore.from_conn_string(DB_URI) as store:
    store.setup()

    graph = builder.compile(store=store)

    config = {"configurable": {"user_id": "u2"}}

    graph.invoke({"messages": [{"role": "user", "content": "Hi, my name is amir"}]}, config)
    graph.invoke({"messages": [{"role": "user", "content": "I am AI/ML engineer"}]}, config)

    out = graph.invoke({"messages": [{"role": "user", "content": "Explain GenAI simply"}]}, config)
    print(out["messages"][-1].content)

    print("\n--- Stored Memories (from Postgres) ---")
    for it in store.search(("user", "u2", "details")):
        print(it.value["data"])

Hi Amir, I'm glad you're interested in learning more about GenAI. As an AI/ML engineer, you likely have a solid foundation in artificial intelligence and machine learning concepts. GenAI, short for General Artificial Intelligence, refers to a type of AI that can perform any intellectual task that a human can. It's a broad and ambitious goal, aiming to create machines that can think, learn, and apply knowledge like humans do.

Think of GenAI like a super-intelligent assistant that can understand and respond to complex questions, learn from experiences, and even make decisions autonomously. It's a significant step beyond narrow or specialized AI, which is designed to excel in a specific domain, like image recognition or language translation.

In the context of your work, Amir, GenAI could potentially revolutionize the way you approach AI/ML projects, enabling you to create more versatile and human-like systems. I'd love to explore this topic further with you.

Here are three potential fo

## Check Persistence

In [21]:
from langgraph.store.postgres import PostgresStore

with PostgresStore.from_conn_string(DB_URI) as store:
    ns = ("user", "u2", "details")
    items = store.search(ns)

for it in items:
    print(it.value["data"])


User asked about GenAI explanation
User asked about GenAI
I am an AI/ML engineer
My name is Amir
